# Preparation

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
import networkx as nx
import numpy as np
import re
from adjustText import adjust_text
from bertopic import BERTopic

# Upload files

In [ ]:
output_path = 'Results_jan2026/'
topic_model = BERTopic.load(output_path + 'bertopic_dpi_reduced_6jan2026')
topics = pd.read_excel(output_path + 'bertopic_dpi_reduced_7jan2026.xlsx')
excerpts_df = pd.read_csv(output_path + 'excerpts_dpi_6jan2026.csv')

In [ ]:
len(excerpts_df)

In [ ]:
# Display the first few rows of the dataframe
topics.head()

In [ ]:
topics['DPI_Classification_Victo'] = topics['DPI_Classification_Victo'].str.capitalize()
topics['DPI_Classification_Victo'].value_counts()

In [ ]:
topics[topics.DPI_Classification_Victo == 'Safeguarding']['Representation'].tolist()

In [ ]:
excerpts_df['Topic'] = topic_model.get_document_info(excerpts_df['cleaned_excerpts'].to_list())['Topic']

In [ ]:
excerpts_df

In [ ]:
from wordcloud import WordCloud

# Combine all representation keywords for promoting topics, excluding specific keywords
promoting_keywords = ' '.join([
    keyword for sublist in topics[topics['DPI_Classification_Victo'] == 'Promoting']['Representation'].apply(eval).tolist()
    for keyword in sublist if keyword.lower().strip()
])

# Combine all representation keywords for safeguarding topics
safeguarding_keywords = ' '.join([
    keyword for sublist in topics[topics['DPI_Classification_Victo'] == 'Safeguarding']['Representation'].apply(eval).tolist()
    for keyword in sublist if keyword.lower().strip()
])

# Generate word clouds
promoting_wordcloud = WordCloud(width=800, height=400, background_color='white').generate(promoting_keywords)
safeguarding_wordcloud = WordCloud(width=800, height=400, background_color='white').generate(safeguarding_keywords)

# Plot the word clouds in different plots
plt.figure(figsize=(16, 8))

# Plot promoting word cloud
plt.subplot(1, 1, 1)
plt.imshow(promoting_wordcloud, interpolation='bilinear')
plt.title('Promoting Topics Word Cloud')
plt.axis('off')
plt.show()

# Plot safeguarding word cloud
plt.figure(figsize=(16, 8))
plt.subplot(1, 1, 1)
plt.imshow(safeguarding_wordcloud, interpolation='bilinear')
plt.title('Safeguarding Topics Word Cloud')
plt.axis('off')
plt.show()

In [ ]:
topics['Representation'] 

In [ ]:
promoting_topics = topics[topics['DPI_Classification_Victo'] == 'Promoting']['Topic'].tolist()

safeguarding_topics = topics[topics['DPI_Classification_Victo'] == 'Safeguarding']['Topic'].tolist()

# Print the lists of topics
print(promoting_topics)
print(safeguarding_topics)

In [ ]:
# Filter excerpts for promoting and safeguarding topics, convert to lowercase
promoting_excerpts_text = ' '.join(
    excerpts_df[excerpts_df['Topic'].isin(promoting_topics)]['cleaned_excerpts'].str.lower().tolist()
)
safeguarding_excerpts_text = ' '.join(
    excerpts_df[excerpts_df['Topic'].isin(safeguarding_topics)]['cleaned_excerpts'].str.lower().tolist()
)

# Generate word clouds
promoting_excerpts_wordcloud = WordCloud(width=800, height=400, background_color='white').generate(promoting_excerpts_text)
safeguarding_excerpts_wordcloud = WordCloud(width=800, height=400, background_color='white').generate(safeguarding_excerpts_text)

# Plot the word clouds
plt.figure(figsize=(16, 8))
plt.subplot(1, 2, 1)
plt.imshow(promoting_excerpts_wordcloud, interpolation='bilinear')
plt.title('Promoting Topics Excerpts Word Cloud')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(safeguarding_excerpts_wordcloud, interpolation='bilinear')
plt.title('Safeguarding Topics Excerpts Word Cloud')
plt.axis('off')

plt.show()


In [ ]:
#Number of topics in promoiting
len(promoting_topics)

In [ ]:
#Number of excerpts in promoting topics
topics[topics['DPI_Classification_Victo'] == 'Promoting']['Count'].sum()

In [ ]:
#Number of topics in safeguarding
len(safeguarding_topics)

In [ ]:
#Number of excerpts in safeguarding topics
topics[topics['DPI_Classification_Victo'] == 'Safeguarding']['Count'].sum()

In [ ]:
# Classify topics in excerpts as promoting or safeguarding
excerpts_df['Promoting'] = excerpts_df['Topic'].apply(lambda x: x in promoting_topics)

excerpts_df['Safeguarding'] = excerpts_df['Topic'].apply(lambda x: x in safeguarding_topics)

# Filter excerpts based on promoting or safeguarding topics
promoting_excerpts = excerpts_df[excerpts_df['Promoting']]
safeguarding_excerpts = excerpts_df[excerpts_df['Safeguarding']]

In [ ]:
# Display the filtered excerpts
promoting_excerpts.head()

In [ ]:
safeguarding_excerpts.head()

In [ ]:
# Group by document_name and count the number of promoting, safeguarding topics, and total excerpts
grouped_counts = excerpts_df.groupby('document_name').agg(
    promoting_count=('Promoting', 'sum'),
    safeguarding_count=('Safeguarding', 'sum'),
    total_excerpts=('document_name', 'size')
).reset_index()

# Display the grouped counts
grouped_counts.head()

In [ ]:
download_report = 'PDFs_May_2025/download_report_v2.xlsx'
overton_data = pd.read_excel(download_report)

# Display the first few rows of the dataframe
overton_data.head()

In [ ]:
# Create the Filename column by combining 'Downloaded Filename' and 'Manual Downloaded Filename'
overton_data['Filename'] = overton_data['Downloaded Filename'].combine_first(
    overton_data['Manual Downloaded Filename'].apply(lambda x: f"{x}.pdf" if pd.notnull(x) else x)
)

In [ ]:
overton_data.Filename

In [ ]:
grouped_counts.document_name

In [ ]:
# Merge the excerpts dataframe with the overton dataframe on the new column and the Unnamed: 0 column
documents_topics = pd.merge(grouped_counts, overton_data, left_on='document_name', right_on='Filename', how='left')

# Convert the Published column to datetime
documents_topics['Published'] = pd.to_datetime(documents_topics['Published'], errors='coerce')

# Extract the year from the Published column
documents_topics['Published Year'] = documents_topics['Published'].dt.year

# Display the merged dataframe
documents_topics.head()

In [ ]:
documents_topics

In [ ]:
excerpts_documents = pd.merge(excerpts_df, documents_topics, left_on='document_name', right_on='document_name', how='left')

excerpts_documents.head()

In [ ]:
excerpts_documents.columns

In [ ]:
def get_yearly_counts(excerpts_documents):
    yearly_counts = excerpts_documents.groupby('Published Year', as_index=False).agg(
        promoting_count=('Promoting', 'sum'),
        safeguarding_count=('Safeguarding', 'sum'),
        total_excerpts=('document_name', 'size')
    )
    yearly_counts['promoting_percentage'] = yearly_counts['promoting_count'] / yearly_counts['total_excerpts']
    yearly_counts['safeguarding_percentage'] = yearly_counts['safeguarding_count'] / yearly_counts['total_excerpts']
    return yearly_counts

In [ ]:
def plot_yearly_promoting_safeguarding(excerpts_documents, label, exclude_years=None):
    yearly_counts = get_yearly_counts(excerpts_documents)

    # Plot counts
    plt.figure(figsize=(14, 8))
    plt.plot(yearly_counts['Published Year'], yearly_counts['promoting_count'], marker='o', label='Promoting Excerpts')
    plt.plot(yearly_counts['Published Year'], yearly_counts['safeguarding_count'], marker='o', label='Safeguarding Excerpts')
    plt.xlabel('Published Year')
    plt.ylabel('Number of Excerpts')
    plt.title(f'Number of Promoting and Safeguarding Excerpts per Year - {label}')
    plt.legend(title='Excerpts Type')
    plt.grid(True)
    plt.show()

    # Filter years if specified
    if exclude_years is not None:
        filtered_yearly_counts = yearly_counts[~yearly_counts['Published Year'].isin(exclude_years)]
    else:
        filtered_yearly_counts = yearly_counts

    # Plot percentages
    plt.figure(figsize=(14, 8))
    plt.plot(filtered_yearly_counts['Published Year'], filtered_yearly_counts['promoting_percentage'], marker='o', label='Promoting Percentage')
    plt.plot(filtered_yearly_counts['Published Year'], filtered_yearly_counts['safeguarding_percentage'], marker='o', label='Safeguarding Percentage')
    plt.xlabel('Published Year')
    plt.ylabel('Percentage of Excerpts')
    plt.title(f'Percentage of Promoting and Safeguarding Excerpts per Year - {label}')
    plt.legend(title='Type')
    plt.grid(True)
    plt.show()


In [ ]:
plot_yearly_promoting_safeguarding(excerpts_documents, '', exclude_years=None)

In [ ]:
excerpts_documents['Source Organisation Type'].value_counts()

In [ ]:
excerpts_documents_tt = excerpts_documents[excerpts_documents['Source Organisation Type'] == 'Think Tank']
excerpts_documents_igo = excerpts_documents[excerpts_documents['Source Organisation Type'] == 'IGO']
excerpts_documents_gov = excerpts_documents[excerpts_documents['Source Organisation Type'] == 'Government']
excerpts_documents_ngo = excerpts_documents[excerpts_documents['Source Organisation Type'] == 'NGO']
excerpts_documents_leg = excerpts_documents[excerpts_documents['Source Organisation Type'] == 'Legislative Body']

In [ ]:
plot_yearly_promoting_safeguarding(excerpts_documents_tt, 'Think Tank', exclude_years=None)

In [ ]:
plot_yearly_promoting_safeguarding(excerpts_documents_igo, 'IGO', exclude_years=None)

In [ ]:
plot_yearly_promoting_safeguarding(excerpts_documents_gov, 'Government', exclude_years=None)

In [ ]:
plot_yearly_promoting_safeguarding(excerpts_documents_ngo, 'NGO', exclude_years=None)

In [ ]:
plot_yearly_promoting_safeguarding(excerpts_documents_leg, 'Legislative Body', exclude_years=None)

In [ ]:
yearly_counts = get_yearly_counts(excerpts_documents)
yearly_counts_tt = get_yearly_counts(excerpts_documents_tt)
yearly_counts_igo = get_yearly_counts(excerpts_documents_igo)
yearly_counts_gov = get_yearly_counts(excerpts_documents_gov)   
yearly_counts_ngo = get_yearly_counts(excerpts_documents_ngo)
yearly_counts_leg = get_yearly_counts(excerpts_documents_leg)

In [ ]:
plt.figure(figsize=(14, 8))
plt.plot(yearly_counts_tt['Published Year'], yearly_counts_tt['promoting_percentage'], marker='o', label='Think Tank')
plt.plot(yearly_counts_igo['Published Year'], yearly_counts_igo['promoting_percentage'], marker='o', label='IGO')
plt.plot(yearly_counts_gov['Published Year'], yearly_counts_gov['promoting_percentage'], marker='o', label='Government')
plt.plot(yearly_counts_ngo['Published Year'], yearly_counts_ngo['promoting_percentage'], marker='o', label='NGO')
plt.plot(yearly_counts_leg['Published Year'], yearly_counts_leg['promoting_percentage'], marker='o', label='Legislative Body')
plt.plot(yearly_counts['Published Year'], yearly_counts['promoting_percentage'], marker='o', color='black', linewidth=2.5, label='Total')

plt.xlabel('Published Year')
plt.ylabel('Promoting Percentage')
plt.title('Promoting Percentage per Year by Organisation Type')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(14, 8))
plt.plot(yearly_counts_tt['Published Year'], yearly_counts_tt['safeguarding_percentage'], marker='o', label='Think Tank')
plt.plot(yearly_counts_igo['Published Year'], yearly_counts_igo['safeguarding_percentage'], marker='o', label='IGO')
plt.plot(yearly_counts_gov['Published Year'], yearly_counts_gov['safeguarding_percentage'], marker='o', label='Government')
plt.plot(yearly_counts_ngo['Published Year'], yearly_counts_ngo['safeguarding_percentage'], marker='o', label='NGO')
plt.plot(yearly_counts_leg['Published Year'], yearly_counts_leg['safeguarding_percentage'], marker='o', label='Legislative Body')
plt.plot(yearly_counts['Published Year'], yearly_counts['safeguarding_percentage'], marker='o', color='black', linewidth=2.5, label='Total')

plt.xlabel('Published Year')
plt.ylabel('Safeguarding Percentage')
plt.title('Safeguarding Percentage per Year by Organisation Type')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(14, 8))

bar_width = 0.13
years = yearly_counts['Published Year']
x = np.arange(len(years))

plt.bar(x - 2*bar_width, yearly_counts_tt.set_index('Published Year').reindex(years)['promoting_percentage'], width=bar_width, label='Think Tank')
plt.bar(x - bar_width, yearly_counts_igo.set_index('Published Year').reindex(years)['promoting_percentage'], width=bar_width, label='IGO')
plt.bar(x, yearly_counts_gov.set_index('Published Year').reindex(years)['promoting_percentage'], width=bar_width, label='Government')
plt.bar(x + bar_width, yearly_counts_ngo.set_index('Published Year').reindex(years)['promoting_percentage'], width=bar_width, label='NGO')
plt.bar(x + 2*bar_width, yearly_counts_leg.set_index('Published Year').reindex(years)['promoting_percentage'], width=bar_width, label='Legislative Body')
plt.bar(x + 3*bar_width, yearly_counts['promoting_percentage'], width=bar_width, color='black', label='Total')

plt.xlabel('Published Year')
plt.ylabel('Promoting Percentage')
plt.title('Promoting Percentage per Year by Organisation Type')
plt.xticks(x, years.astype(int))
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
documents_topics['promoting_percentage'] = documents_topics.promoting_count / documents_topics.total_excerpts
documents_topics['safeguarding_percentage'] = documents_topics.safeguarding_count / documents_topics.total_excerpts
documents_topics

In [ ]:
documents_topics.columns

In [ ]:
# Group by Published Year and calculate the average promoting and safeguarding counting
average_counting = documents_topics[~documents_topics['Published Year'].isin([2018, 2019])].groupby('Published Year').agg(
    avg_promoting_counting=('promoting_count', 'mean'),
    avg_safeguarding_counting=('safeguarding_count', 'mean')
).reset_index()

# Plot the data as a line graph
plt.figure(figsize=(14, 8))
plt.plot(average_counting['Published Year'], average_counting['avg_promoting_counting'], marker='o', label='Promoting mentions')
plt.plot(average_counting['Published Year'], average_counting['avg_safeguarding_counting'], marker='o', label='Safeguarding mentions')

plt.xlabel('Published Year')
plt.ylabel('Average metions')
plt.title('Average Promoting and Safeguarding excerpts per Year')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Group by Published Year and calculate the average promoting and safeguarding percentages
average_percentages = documents_topics[~documents_topics['Published Year'].isin([2018, 2019])].groupby('Published Year').agg(
    avg_promoting_percentage=('promoting_percentage', 'mean'),
    avg_safeguarding_percentage=('safeguarding_percentage', 'mean')
).reset_index()

# Plot the data as a line graph
plt.figure(figsize=(14, 8))
plt.plot(average_percentages['Published Year'], average_percentages['avg_promoting_percentage'], marker='o', label='Average Promoting Percentage')
plt.plot(average_percentages['Published Year'], average_percentages['avg_safeguarding_percentage'], marker='o', label='Average Safeguarding Percentage')

plt.xlabel('Published Year')
plt.ylabel('Average Percentage')
plt.title('Average Promoting and Safeguarding Percentage per Year (Excluding 2018 and 2019)')
plt.legend(title='Percentage Type')
plt.grid(True)
plt.show()


In [ ]:
documents_topics[documents_topics['Published Year'] == 2019]

In [ ]:
#Group by Published Year and calculate the average promoting and safeguarding percentages
documents_topics_tt = documents_topics[documents_topics['Source Organisation Type'] == 'Think Tank']
documents_topics_gov = documents_topics[documents_topics['Source Organisation Type'] == 'Government']
documents_topics_igo = documents_topics[documents_topics['Source Organisation Type'] == 'IGO']
documents_topics_ngo = documents_topics[documents_topics['Source Organisation Type'] == 'NGO']
documents_topics_leg = documents_topics[documents_topics['Source Organisation Type'] == 'Legislative Body']


average_percentages = documents_topics_tt.groupby('Published Year').agg(
    avg_promoting_percentage=('promoting_percentage', 'mean'),
    avg_safeguarding_percentage=('safeguarding_percentage', 'mean')
).reset_index()

# Plot the data as a line graph
plt.figure(figsize=(14, 8))
plt.plot(average_percentages['Published Year'], average_percentages['avg_promoting_percentage'], marker='o', label='Average Promoting Percentage')
plt.plot(average_percentages['Published Year'], average_percentages['avg_safeguarding_percentage'], marker='o', label='Average Safeguarding Percentage')

plt.xlabel('Published Year')
plt.ylabel('Average Percentage')
plt.title('Average Promoting and Safeguarding Percentage per Year - Think Tanks')
plt.legend(title='Percentage Type')
plt.grid(True)
plt.show()

In [ ]:
average_percentages = documents_topics_gov.groupby('Published Year').agg(
    avg_promoting_percentage=('promoting_percentage', 'mean'),
    avg_safeguarding_percentage=('safeguarding_percentage', 'mean')
).reset_index()

# Plot the data as a line graph
plt.figure(figsize=(14, 8))
plt.plot(average_percentages['Published Year'], average_percentages['avg_promoting_percentage'], marker='o', label='Average Promoting Percentage')
plt.plot(average_percentages['Published Year'], average_percentages['avg_safeguarding_percentage'], marker='o', label='Average Safeguarding Percentage')

plt.xlabel('Published Year')
plt.ylabel('Average Percentage')
plt.title('Average Promoting and Safeguarding Percentage per Year - Government')
plt.legend(title='Percentage Type')
plt.grid(True)
plt.show()

In [ ]:
average_percentages = documents_topics_igo.groupby('Published Year').agg(
    avg_promoting_percentage=('promoting_percentage', 'mean'),
    avg_safeguarding_percentage=('safeguarding_percentage', 'mean')
).reset_index()

# Plot the data as a line graph
plt.figure(figsize=(14, 8))
plt.plot(average_percentages['Published Year'], average_percentages['avg_promoting_percentage'], marker='o', label='Average Promoting Percentage')
plt.plot(average_percentages['Published Year'], average_percentages['avg_safeguarding_percentage'], marker='o', label='Average Safeguarding Percentage')

plt.xlabel('Published Year')
plt.ylabel('Average Percentage')
plt.title('Average Promoting and Safeguarding Percentage per Year - IGO')
plt.legend(title='Percentage Type')
plt.grid(True)
plt.show()

In [ ]:
average_percentages = documents_topics_ngo.groupby('Published Year').agg(
    avg_promoting_percentage=('promoting_percentage', 'mean'),
    avg_safeguarding_percentage=('safeguarding_percentage', 'mean')
).reset_index()

# Plot the data as a line graph
plt.figure(figsize=(14, 8))
plt.plot(average_percentages['Published Year'], average_percentages['avg_promoting_percentage'], marker='o', label='Average Promoting Percentage')
plt.plot(average_percentages['Published Year'], average_percentages['avg_safeguarding_percentage'], marker='o', label='Average Safeguarding Percentage')

plt.xlabel('Published Year')
plt.ylabel('Average Percentage')
plt.title('Average Promoting and Safeguarding Percentage per Year - NGO')
plt.legend(title='Percentage Type')
plt.grid(True)
plt.show()

In [ ]:
average_percentages = documents_topics_leg.groupby('Published Year').agg(
    avg_promoting_percentage=('promoting_percentage', 'mean'),
    avg_safeguarding_percentage=('safeguarding_percentage', 'mean')
).reset_index()

# Plot the data as a line graph
plt.figure(figsize=(14, 8))
plt.plot(average_percentages['Published Year'], average_percentages['avg_promoting_percentage'], marker='o', label='Average Promoting Percentage')
plt.plot(average_percentages['Published Year'], average_percentages['avg_safeguarding_percentage'], marker='o', label='Average Safeguarding Percentage')

plt.xlabel('Published Year')
plt.ylabel('Average Percentage')
plt.title('Average Promoting and Safeguarding Percentage per Year - Legislative Body')
plt.legend(title='Percentage Type')
plt.grid(True)
plt.show()

In [ ]:
excerpts_documents.columns

In [ ]:
# Merge topics.Topic_Label into excerpts_documents based on the Topic column
excerpts_documents = excerpts_documents.merge(
    topics[['Topic', 'Topic_Label']],
    left_on='Topic',
    right_on='Topic',
    how='left'
)

In [ ]:
# Get top 10 topics by total count
top10_topics = topics.nlargest(10, 'Count')

# Prepare a dataframe with excerpts and their year and topic label
excerpts_with_year = excerpts_documents[['Published Year', 'Topic', 'Topic_Label']].dropna(subset=['Published Year', 'Topic'])

# Filter for top 10 topics only
excerpts_top10 = excerpts_with_year[excerpts_with_year['Topic'].isin(top10_topics['Topic'])]

# Group by year and topic label, count occurrences
topic_year_counts = excerpts_top10.groupby(['Published Year', 'Topic_Label']).size().unstack(fill_value=0)

# Plot
plt.figure(figsize=(16, 8))
topic_year_counts.plot(kind='line', stacked=False, ax=plt.gca())
plt.xlabel('Published Year')
plt.ylabel('Excerpt Count')
plt.title('Top 10 Topics Count per Year')
plt.legend(title='Topic Label', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Exclude 2025 from the plot
topic_year_counts_excl_2025 = topic_year_counts.drop(index=2025.0, errors='ignore')

plt.figure(figsize=(16, 8))
topic_year_counts_excl_2025.plot(kind='line', stacked=False, ax=plt.gca())
plt.xlabel('Published Year')
plt.ylabel('Excerpt Count')
plt.title('Top 10 Topics Count per Year (Excluding 2025)')
plt.legend(title='Topic Label', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Get the 5 most frequent topics
top5_topics = topics.nlargest(5, 'Count')

# Prepare a dataframe with excerpts, their source, and topic label
excerpts_with_source = excerpts_documents[['Source Organisation Type', 'Topic', 'Topic_Label']].dropna(subset=['Source Organisation Type', 'Topic'])

# Filter for top 5 topics only
excerpts_top5 = excerpts_with_source[excerpts_with_source['Topic'].isin(top5_topics['Topic'])]

# Group by source and topic label, count occurrences
source_topic_counts = excerpts_top5.groupby(['Source Organisation Type', 'Topic_Label']).size().unstack(fill_value=0)

# Plot
source_topic_counts.plot(kind='bar', figsize=(14, 8))
plt.xlabel('Source Organisation Type')
plt.ylabel('Excerpt Count')
plt.title('Top 5 Topics Count per Source Organisation Type')
plt.legend(title='Topic Label', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
excerpts_documents

In [ ]:
# Filter documents that have at least one promoting topic
promoting_documents = documents_topics[documents_topics['promoting_count'] > 0]

# Group by Published Year and count the number of promoting documents
yearly_promoting_counts = promoting_documents.groupby('Published Year').size().reset_index(name='document_count')

# Filter documents that have at least one safeguarding topic
safeguarding_documents = documents_topics[documents_topics['safeguarding_count'] > 0]

# Group by Published Year and count the number of safeguarding documents
yearly_safeguarding_counts = safeguarding_documents.groupby('Published Year').size().reset_index(name='document_count')

# Group by Published Year and count the total number of documents
yearly_total_counts = documents_topics.groupby('Published Year').size().reset_index(name='document_count')

# Plot the data as line graphs
plt.figure(figsize=(14, 8))

# Plot promoting documents
plt.plot(yearly_promoting_counts['Published Year'], yearly_promoting_counts['document_count'], marker='o', label='Documents mentioning Promoting Topics')

# Plot safeguarding documents
plt.plot(yearly_safeguarding_counts['Published Year'], yearly_safeguarding_counts['document_count'], marker='o', label='Documents mentioning Safeguarding Topics')

# Plot total documents
plt.plot(yearly_total_counts['Published Year'], yearly_total_counts['document_count'], marker='.', color='black', linestyle=':', label='Total Documents')

plt.xlabel('Published Year')
plt.ylabel('Number of Documents')
plt.title('Number of Documents mentioning promoting or safeguarding topics per Year')
plt.legend()
plt.grid(True)
plt.show()